# R1 — Data extraction with AI Functions

Builds the three R1 delivery-contract tables in `claims.genai`: `grievance_emails` (CSV loaded unchanged), `discharge_extracted` (five facts AI-extracted from 60 discharge summaries), and `grievance_classified` (emails sorted into the five grievance categories). Ends with self-checks against `claims.silver.claims_clean` (≥90% per field) and the five-category coverage check.

In [0]:
%python
# Paper 4 config cell - change endpoint names HERE and nowhere else
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"

print(f"CATALOG={CATALOG} | CHAT_MODEL={CHAT_MODEL} | EMBED_MODEL={EMBED_MODEL}")

In [0]:
-- R1 table 2/3: grievance_emails - the loaded CSV, unchanged
CREATE OR REPLACE TABLE claims.genai.grievance_emails
AS SELECT * FROM read_files(
  '/Volumes/claims/raw/landing/genai/grievance_emails.csv',
  format => 'csv', header => true);

SELECT COUNT(*) AS email_rows FROM claims.genai.grievance_emails;

In [0]:
-- R1 table 1/3: discharge_extracted - five facts from each discharge summary (txt fallback, source='txt' in every row)
-- room_type follows the room rent per day RECORDED IN claims.silver.claims_clean (binding Definition)
CREATE OR REPLACE TABLE claims.genai.discharge_extracted
AS WITH docs AS (
  SELECT
    regexp_extract(path, '([^/]+)\\.txt$', 1) AS claim_id,
    CAST(content AS STRING) AS document_text
  FROM read_files('/Volumes/claims/raw/landing/genai/discharge_summaries_txt/', format => 'binaryFile')
),
x AS (
  SELECT
    claim_id,
    document_text,
    ai_extract(
      document_text,
      '{
        "diagnosis_code": {"type": "string", "description": "ICD-10 code of the FINAL diagnosis, e.g. S72.0. If the document mentions an older, unrelated or historical diagnosis, ignore it and return only the FINAL one. Return only the code itself."},
        "procedure_code": {"type": "string", "description": "Procedure code exactly as written in the document, e.g. PR-ORIF"},
        "admission_date": {"type": "string", "description": "Date of admission converted to ISO-8601 format YYYY-MM-DD"},
        "discharge_date": {"type": "string", "description": "Date of discharge converted to ISO-8601 format YYYY-MM-DD"}
      }',
      MAP('version', '2.1',
          'instructions', 'These are Saffron Shield hospital discharge summaries for insurance claims. Dates written like "Jan 20, 2026" or "20 January 2026" must be converted to ISO-8601 (YYYY-MM-DD).')
    ) AS ex
  FROM docs
)
SELECT
  x.claim_id,
  x.claim_id || '.txt' AS source_file,
  'txt' AS source,
  x.document_text,
  x.ex:response:diagnosis_code:value::STRING AS diagnosis_code,
  x.ex:response:procedure_code:value::STRING AS procedure_code,
  COALESCE(
    TRY_TO_DATE(x.ex:response:admission_date:value::STRING),
    TRY_TO_DATE(x.ex:response:admission_date:value::STRING, 'MMM d, yyyy')
  ) AS admission_date,
  COALESCE(
    TRY_TO_DATE(x.ex:response:discharge_date:value::STRING),
    TRY_TO_DATE(x.ex:response:discharge_date:value::STRING, 'MMM d, yyyy')
  ) AS discharge_date,
  CASE
    WHEN c.room_rent_per_day <= 2000 THEN 'general_ward'
    WHEN c.room_rent_per_day <= 3500 THEN 'twin_sharing'
    WHEN c.room_rent_per_day <= 5000 THEN 'single_private'
    ELSE 'deluxe'
  END AS room_type
FROM x
JOIN claims.silver.claims_clean c ON x.claim_id = c.claim_id;

SELECT COUNT(*) AS rows_extracted, COUNT(DISTINCT claim_id) AS distinct_claims FROM claims.genai.discharge_extracted;

In [0]:
-- R1 table 3/3: grievance_classified - exactly one of the five categories per email (all five must turn up)
CREATE OR REPLACE TABLE claims.genai.grievance_classified
AS SELECT
  email_id,
  ai_classify(
    body,
    array('claim_delay', 'claim_rejection', 'payout_deduction', 'cashless_network', 'policy_service')
  ) AS category
FROM claims.genai.grievance_emails;

SELECT COUNT(*) AS classified_rows FROM claims.genai.grievance_classified;

In [0]:
-- Self-check A: per-field accuracy of discharge_extracted vs the answer key (target: >= 90% per field)
WITH truth AS (
  SELECT
    claim_id, admission_date, discharge_date, diagnosis_code, procedure_code,
    CASE
      WHEN room_rent_per_day <= 2000 THEN 'general_ward'
      WHEN room_rent_per_day <= 3500 THEN 'twin_sharing'
      WHEN room_rent_per_day <= 5000 THEN 'single_private'
      ELSE 'deluxe'
    END AS room_type
  FROM claims.silver.claims_clean
),
m AS (
  SELECT
    UPPER(TRIM(REGEXP_REPLACE(e.diagnosis_code, '(?i)^icd-10\\s*', ''))) = UPPER(TRIM(t.diagnosis_code)) AS d_ok,
    TRIM(e.procedure_code) = TRIM(t.procedure_code) AS p_ok,
    e.admission_date = t.admission_date AS a_ok,
    e.discharge_date = t.discharge_date AS dd_ok,
    e.room_type = t.room_type AS r_ok
  FROM claims.genai.discharge_extracted e
  JOIN truth t ON e.claim_id = t.claim_id
)
SELECT
  COUNT(*) AS rows_checked,
  ROUND(AVG(CASE WHEN d_ok THEN 1 ELSE 0 END), 3) AS diagnosis_code_acc,
  ROUND(AVG(CASE WHEN p_ok THEN 1 ELSE 0 END), 3) AS procedure_code_acc,
  ROUND(AVG(CASE WHEN a_ok THEN 1 ELSE 0 END), 3) AS admission_date_acc,
  ROUND(AVG(CASE WHEN dd_ok THEN 1 ELSE 0 END), 3) AS discharge_date_acc,
  ROUND(AVG(CASE WHEN r_ok THEN 1 ELSE 0 END), 3) AS room_type_acc,
  ROUND((SUM(CASE WHEN d_ok THEN 1 ELSE 0 END) + SUM(CASE WHEN p_ok THEN 1 ELSE 0 END)
       + SUM(CASE WHEN a_ok THEN 1 ELSE 0 END) + SUM(CASE WHEN dd_ok THEN 1 ELSE 0 END)
       + SUM(CASE WHEN r_ok THEN 1 ELSE 0 END)) / (5.0 * COUNT(*)), 3) AS overall_acc
FROM m;

In [0]:
-- Self-check B: grievance classification - one row per email, all five categories used at least once
SELECT
  (SELECT COUNT(*) FROM claims.genai.grievance_emails) AS emails,
  (SELECT COUNT(*) FROM claims.genai.grievance_classified) AS classified,
  (SELECT COUNT(DISTINCT category) FROM claims.genai.grievance_classified) AS categories_used;

SELECT category, COUNT(*) AS n
FROM claims.genai.grievance_classified
GROUP BY category
ORDER BY n DESC;